### This notebook serves as a tutorial and a way to regenerate the results and SHAP matrices used in the scCont paper. The paper is organized into the five steps outlined in Figure 1.

Importing packages: the pipeline itself is provided by the `sccont` package (`pip install sccont`). The remaining imports are only needed for the latent-level analysis and figure cells at the end of this notebook. See `requirements.txt` for the exact package versions used in the paper.

In [ ]:
import json
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import seaborn as sns
import shap
from scipy.stats import pearsonr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

import sccont
print('sccont', sccont.__version__)

Setting Random Seed: All results were generated with random seed = 42

In [ ]:
seed = 42
sccont.set_seeds(seed)
device = sccont.get_device()
print('device:', device)

In [ ]:
#Pathway file for saving
file_path = 'MCF10A_TGFB1/'

Loading the data: scCont works on an `AnnData` object (cells x genes). Any per-cell labels (timepoint, condition, batch, ...) go in `adata.obs`; they are carried through every step and used for interpretation and figures, but training itself is unsupervised and never reads them. The repository's genes x cells tables are converted with `sccont.to_anndata`; here the timepoint is parsed from cell names of the form `V(cell_number)_T(timepoint)`, but you can attach labels from any source (e.g. `adata.obs['condition'] = ...`).

Normalizing the data: if you are using a dataset from this repository or your dataset is already normalized, set the `normalize` flag to False. Otherwise `sccont.preprocess` runs QC filtering, Scrublet doublet removal, library-size normalization, log1p, selection of 3000 highly variable genes, regression of `percent_mito`, `n_counts` and cell-cycle scores, and scaling. Every step has its own switch (`help(sccont.preprocess)`). We use the MCF10A TGFB1 dataset as an example.

In [ ]:
normalize = True

scRNA_data = sccont.load_expression_matrix(file_path + 'GSE200981_scRNAseq_processed.tsv')  # genes x cells
adata = sccont.to_anndata(scRNA_data, timepoint=sccont.timepoints_from_columns(scRNA_data.columns))

if normalize:
    adata = sccont.preprocess(adata, plot_qc=True)   # obs labels are preserved

gene_names = list(adata.var_names)
X_genes = adata.X                                    # cells x genes
adata

# A) kNN Pair Selection

Here, we apply a k-Nearest Neighbor graph (k=3) is applied to the normalized gene expression space to identify positive pairs to preserve local topology. PCA is used here to reduce noise in the data.

In [ ]:
pairs = sccont.get_knn_pairs(adata, k=3)  # the paper uses k=3

# B) Contrastive Training

Training uses `sccont.Encoder` (input → 2048 → 1024 → 64 → 32 latent features), `sccont.Projector` (32 → 32 → 16) and `sccont.InfoNCELoss` (temperature 0.20). `sccont.train_contrastive` samples 256 positive pairs per step and optimises with Adam (lr = 1e-4). Training time depends on the dataset; 20 000 steps were used for all paper results.

In [ ]:
epochs = 20000

encoder, projector, loss_sims = sccont.train_contrastive(
    adata, pairs,
    latent_dim=32, proj_dim=16,
    epochs=epochs, batch_size=256, lr=1e-4, temperature=0.20,
    device=device, seed=seed,
)

In [ ]:
sccont.pl.training_loss(loss_sims);

Encode all cells: `sccont.embed` writes the latent features to `adata.obsm['X_sccont']` (and their PCA to `adata.obsm['X_sccont_pca']`). `sccont.latent_anndata` gives a view with the latent features as variables and all `obs` labels copied, which stage C and the figure code below work on. The encoder weights are saved alongside.

In [ ]:
sccont.embed(encoder, adata, device=device)
adata_latent = sccont.latent_anndata(adata)
sccont.save_encoder(encoder, file_path + 'encoder.pth')

In [ ]:
adata_latent.write(file_path + 'latent_embeddings.h5ad')

In [ ]:
adata_latent = sc.read_h5ad(file_path + 'latent_embeddings.h5ad')

# C) Signal Identification via Spatial Clustering

Spatial grouping of scCont latents to identify similar latents with similar spatial patterns

In [ ]:
sccont.pl.elbow(adata, max_k=12, bins=40);

Use the above elbow plot to adjust number of clusters

In [ ]:
groups, _ = sccont.group_spatially_similar_latents(adata, n_groups=3, bins=40)
adata_latent.uns['sccont'] = dict(adata.uns['sccont'])   # keep the latent view in sync

Overview of the latent space with `sccont.pl`: cells coloured by any `obs` label or by a latent feature, every latent feature's activation ordered by spatial cluster, the correlation-distance matrix behind the clustering, and which latent features track the label.

In [ ]:
sccont.pl.latent_embedding(adata, color='timepoint')
sccont.pl.latent_embedding(adata, color=7)
sccont.pl.latent_grid(adata)
sccont.pl.latent_distance_heatmap(adata)
sccont.pl.latent_label_association(adata, 'timepoint');

Plotting clusters

In [ ]:
adata_group0 = adata_latent[:, sccont.latents_in_group(groups, 0)]
adata_group1 = adata_latent[:, sccont.latents_in_group(groups, 1)]
adata_group2 = adata_latent[:, sccont.latents_in_group(groups, 2)]

sc.pp.pca(adata_group0)
sc.pp.pca(adata_group1)
sc.pp.pca(adata_group2)

Package version of Figure 2: per spatial cluster, PCA of that cluster's latent features with a trajectory through the per-timepoint centroids; `split_after` draws a two-branch split after the given category. The next cell is the original manuscript code kept for exact reproduction.

In [ ]:
sccont.pl.group_trajectories(adata, groupby='timepoint', split_after='T2');

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# 1. Group the objects and titles
adata_groups = [adata_group0, adata_group1, adata_group2]
titles = ['Cluster 0', 'Cluster 1', 'Cluster 2']
marker_styles = ['o', '^', 's'] 

# Set the timepoint value where the split begins (e.g., t2)
bifurcation_time = 2 

# 2. Create the figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Global min/max for consistent 'magma' coloring
all_timepoints = pd.concat([adata.obs['timepoint'] for adata in adata_groups])
if all_timepoints.dtype == 'object' or all_timepoints.dtype.name == 'category':
    all_timepoints_numeric = all_timepoints.astype('category').cat.codes
else:
    all_timepoints_numeric = all_timepoints.values
vmin, vmax = all_timepoints_numeric.min(), all_timepoints_numeric.max()

# 3. Loop through each adata
for i, (adata_g, ax) in enumerate(zip(adata_groups, axes)):
    
    pca_coords = adata_g.obsm['X_pca'][:, :2]
    timepoints = adata_g.obs['timepoint']

    if timepoints.dtype == 'object' or timepoints.dtype.name == 'category':
        color_values = timepoints.astype('category').cat.codes.values
    else:
        color_values = timepoints.values

    # A. Background Scatter 
    ax.scatter(
        pca_coords[:, 0],
        pca_coords[:, 1],
        c=color_values,
        cmap='magma', 
        vmin=vmin, vmax=vmax,
        s=10,
        alpha=0.3,       
        edgecolors='none',
        zorder=1         
    )

    unique_timepoints = np.unique(color_values)
    unique_timepoints.sort()
    
    # Lists to hold the final marker positions and colors for plotting
    node_x, node_y, node_c = [], [], []
    
    if i == 2:
        # --- CONTINUOUS BIFURCATION LOGIC (Cluster 2) ---
        trunk_x, trunk_y = [], []
        branch1_x, branch1_y = [], []
        branch2_x, branch2_y = [], []
        
        for tp in unique_timepoints:
            mask = (color_values == tp)
            coords = pca_coords[mask]
            
            if tp <= bifurcation_time:
                # 1. Main trunk logic (single centroid)
                cx, cy = coords[:, 0].mean(), coords[:, 1].mean()
                trunk_x.append(cx)
                trunk_y.append(cy)
                
                node_x.append(cx)
                node_y.append(cy)
                node_c.append(tp)
            else:
                # 2. Branch logic using K-Means (k=2)
                kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(coords)
                c1, c2 = kmeans.cluster_centers_
                
                # If this is the first split point, start branches from the tip of the trunk
                if len(branch1_x) == 0 and len(trunk_x) > 0:
                    branch1_x.append(trunk_x[-1])
                    branch1_y.append(trunk_y[-1])
                    branch2_x.append(trunk_x[-1])
                    branch2_y.append(trunk_y[-1])
                
                # Prevent branches from crisscrossing by matching closest nodes
                prev_b1 = np.array([branch1_x[-1], branch1_y[-1]])
                prev_b2 = np.array([branch2_x[-1], branch2_y[-1]])
                
                dist_straight = np.linalg.norm(prev_b1 - c1) + np.linalg.norm(prev_b2 - c2)
                dist_crossed = np.linalg.norm(prev_b1 - c2) + np.linalg.norm(prev_b2 - c1)
                
                if dist_straight <= dist_crossed:
                    next_b1, next_b2 = c1, c2
                else:
                    next_b1, next_b2 = c2, c1
                
                branch1_x.append(next_b1[0])
                branch1_y.append(next_b1[1])
                branch2_x.append(next_b2[0])
                branch2_y.append(next_b2[1])
                
                node_x.extend([next_b1[0], next_b2[0]])
                node_y.extend([next_b1[1], next_b2[1]])
                node_c.extend([tp, tp])
                
        # Draw the trajectory lines
        ax.plot(trunk_x, trunk_y, color='black', linewidth=2, zorder=2)
        if len(branch1_x) > 1:
            ax.plot(branch1_x, branch1_y, color='black', linewidth=2, zorder=2)
            ax.plot(branch2_x, branch2_y, color='black', linewidth=2, zorder=2)

    else:
        # --- STANDARD LOGIC (Clusters 0 & 1) ---
        for tp in unique_timepoints:
            mask = (color_values == tp)
            cx = pca_coords[mask, 0].mean()
            cy = pca_coords[mask, 1].mean()
            
            node_x.append(cx)
            node_y.append(cy)
            node_c.append(tp)
            
        # Draw single trajectory line
        ax.plot(node_x, node_y, color='black', linewidth=2, zorder=2)
    
    # Plot Large Centroid Markers for whichever logic was used
    marker_to_use = marker_styles[i % len(marker_styles)]
    scatter = ax.scatter(
        node_x, node_y,
        c=node_c,
        cmap='magma',
        vmin=vmin, vmax=vmax,
        marker=marker_to_use,
        s=150,              
        edgecolors='black', 
        linewidths=1.5,
        zorder=3            
    )

    #ax.set_title(titles[i]) 
    ax.axis('off')

# 4. Add shared colorbar
#fig.colorbar(scatter, ax=axes.ravel().tolist(), label='Timepoint')
plt.tight_layout()
plt.savefig('Figure2_V5.png', dpi=300, transparent=True)

plt.show()

# D) Network Attribution Analysis

Network attribution analysis via SHAP is performed to learn gene-latent relationships. `sccont.compute_shap_values` wraps `shap.DeepExplainer` with 100 randomly chosen background cells (seed 42) and returns an array of shape (cells, genes, latent features). This will take some time.

In [ ]:
shap_values = sccont.compute_shap_values(encoder, adata, n_background=100, seed=42, device=device)  # cells x genes x latents
np.save(file_path + 'shap_values.npy', shap_values)
adata.varm['sccont_shap_mean_abs'].shape   # mean |SHAP| per gene and latent, stored on adata

# E) Biological Annotation

## Cluster-Level Analysis

Top genes per latent feature are selected by z-score of mean |SHAP| (`sccont.select_top_genes_by_zscore`, threshold 2.57 ≈ p < 0.01). The union over the latent features in each spatial cluster is tested for GO enrichment with g:Profiler (`sccont.enrich_latent_clusters`, background = all genes in the matrix).

In [ ]:
go_results_l = sccont.enrich_latent_clusters(shap_values, gene_names, groups, zscore_threshold=2.57)

GO enrichment results for each cluster can be viewed in the following output file

In [ ]:
sccont.write_go_results_excel(go_results_l, file_path + 'GO_Enrichment_Results.xlsx')

In [ ]:
sccont.pl.go_enrichment(go_results_l, top_n=10);

In [ ]:
#Collecting enriched genes
all_enriched_genes = sccont.collect_enriched_genes(go_results_l)

In [ ]:
#Top 50 driving genes per latent feature (mean |SHAP| over cells)
top_genes_dict = sccont.top_genes_per_latent(shap_values, gene_names, n=50)

#Swapping indices for summary plots
shap_values_swapped = np.moveaxis(shap_values, 2, 0)

latent_idx = 7
shap.summary_plot(shap_values_swapped[latent_idx], features=X_genes, feature_names=gene_names)

The same views with `sccont.pl`: mean |SHAP| of the top genes coloured by direction, the SHAP beeswarm restricted to the top genes, and the distribution of the latent feature per timepoint.

In [ ]:
sccont.pl.top_genes(adata, latent=latent_idx, n=20, shap_values=shap_values)
sccont.pl.shap_beeswarm(adata, shap_values, latent=latent_idx, n=20)
sccont.pl.latent_by_label(adata, latent_idx, 'timepoint');

## Latent-Level Analysis

This part requires some data analysis and literature searching done by the reader. Functions used to analyze latent features, and functions used to generate the plots in the paper are shown here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def plot_gene_shap_sum_and_pca(
    adata, shap_values, gene_list, gene_names, latent_dim_idx=0, invert_shap_sums=False
):
    """
    Returns a PCA plot (colored by summed SHAP values for the selected genes+latent),
    and finds the timepoint where the mean SHAP sum crosses zero.
    
    The colorbar is forced to center at 0 by setting vmin = -max_abs and vmax = +max_abs.
    """

    shap_latent = shap_values[:, :, latent_dim_idx] 
    shap_df = pd.DataFrame(shap_latent, columns=gene_names)
    
    # SHAP sums for the genes of interest
    sum_shap = shap_df[gene_list].sum(axis=1)
    if invert_shap_sums:
        sum_shap *= -1
    adata.obs['sum_shap'] = sum_shap.values

    # PCA coordinates
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(adata.X)

 
    max_val = np.max(np.abs(sum_shap))
    if max_val == 0:
        max_val = 1

    # PCA plot
    fig, ax = plt.subplots(figsize=(5, 4))
    
    sc = ax.scatter(
        X_pca[:,0], 
        X_pca[:,1], 
        c=sum_shap, 
        cmap='coolwarm', 
        s=15, 
        vmin=-max_val, 
        vmax=max_val
    )
    
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label(f"Sum SHAP ({', '.join(gene_list)}) [Latent {latent_dim_idx}]")
    ax.set_xlabel('PCA1')
    ax.set_ylabel('PCA2')
    ax.set_title(f"PCA Colored by SHAP Sum\nLatent Feature {latent_dim_idx}")
    plt.tight_layout()
    
    means = adata.obs.groupby('timepoint')['sum_shap'].mean()
    ordered_timepoints = np.array(means.index)
    mean_vals = means.values
    
    return fig

Driver genes and their direction (positive = high expression increases the feature) are now computed by `sccont.latent_report`, which also attaches the GO terms enriched in the feature's spatial cluster and the timepoints where the feature is high or low. `report.to_prompt()` renders the description we pasted into Claude when curating the functional groups for the paper.

In [ ]:
report = sccont.latent_report(adata, shap_values, 7, go_results=go_results_l, groupby='timepoint', invert=True)
print(report.to_prompt())
report.to_frame().head(10)

In [ ]:
#Individual analysis of latents used to find functional groups
#Sometimes the latents need to be flipped if latent is inversely correlated with time so downstream analysis is easier to understand

shap_values_swapped = np.moveaxis(shap_values, 2, 0)
def get_latent_output(adata_view, latent_idx, all_enriched_genes, flip_latent=False):
    report = sccont.latent_report(adata, shap_values, latent_idx, go_results=go_results_l,
                                  groupby='timepoint', invert=flip_latent)
    latent_genes_set = set(report.genes)

    plot_gene_shap_sum_and_pca(adata_view, shap_values,
                               list(latent_genes_set), gene_names, latent_idx, invert_shap_sums=flip_latent)
    plt.show()

    sccont.pl.shap_beeswarm(adata, shap_values, latent_idx, genes=list(latent_genes_set))

    print(report.to_prompt())
    return latent_genes_set.intersection(all_enriched_genes)

The get_latent_output function was used to evaluate the latent features. This function can be called to assess latent features of interest. The function will also output positive and negative drivers, which can be used to help identify functional groups.

In [ ]:
#Removing Cluster 1 to focus analysis on coherent and bifurcating clusters

emt_latents = [str(i) for i in range(len(groups)) if groups[i] == 0] + [str(i) for i in range(len(groups)) if groups[i] == 2] 
adata_emt = adata_latent[:, emt_latents]

In [ ]:
latent_0_gene_set = get_latent_output(adata_emt, 0, all_enriched_genes)

In [ ]:
latent_5_gene_set = get_latent_output(adata_emt, 5, all_enriched_genes)

In [ ]:
latent_6_gene_set = get_latent_output(adata_emt, 6, all_enriched_genes)

In [ ]:
latent_7_gene_set = get_latent_output(adata_emt, 7, all_enriched_genes, flip_latent=True)

In [ ]:
latent_21_gene_set = get_latent_output(adata_emt, 21, all_enriched_genes)

In [ ]:
latent_25_gene_set = get_latent_output(adata_emt, 25, all_enriched_genes)

In [ ]:
latent_31_gene_set = get_latent_output(adata_emt, 31, all_enriched_genes)

**Optional: LLM-assisted functional groups.** With `pip install "sccont[llm]"` and an `ANTHROPIC_API_KEY`, `sccont.annotate_latents` sends each latent's report to Claude and returns `functional_groups` / `group_to_latent` in the same format as the JSON files loaded below. The shipped JSON files are our curated version of exactly this output; treat a fresh run as a hypothesis to review (see the `rationale` and `confidence` columns of `result.table`).

In [ ]:
import os
if os.environ.get("ANTHROPIC_API_KEY"):
    result = sccont.annotate_latents(
        adata, shap_values, latents=[0, 5, 6, 7, 21, 25, 31],
        go_results=go_results_l, groupby='timepoint', invert_shap={7: True},
        extra_context="MCF10A mammary epithelial cells, TGF-beta1 time course (EMT)",
    )
    display(result.table)
    # result.save(file_path)   # uncomment to overwrite the curated JSON files with this run
else:
    print("ANTHROPIC_API_KEY not set – skipping LLM annotation; using the curated JSON files below.")

We load the json files of our own analysis detailed in the paper. These json files were curated via latent feature annotation based on the enriched genes above. The json files are all dictionaries and do the following:
* functional_groups.json: Mapping of functional group name to genes
* group_to_latent.json: Mapping of functional group to which latent feature it was identified
* invert_shap.json: Mapping of latent to whether the latent sign needs to flipped

In [ ]:
with open(file_path + "functional_groups.json", "r") as file:
    functional_groups = json.load(file)
    
with open(file_path + "group_to_latent.json", "r") as file:
    group_to_latent = json.load(file)
    
with open(file_path + "invert_shap.json", "r") as file:
    invert_shap = json.load(file)
    invert_shap = {int(k): v for k, v in invert_shap.items()}

This is an extraneous file for the MCF10A\_TGFB1 dataset. Since, we detected a bifurcation, a latent_to_bifurcation file is needed. It is important to note that for testing bifurcation, we only used latents in Cluster 2 and did subsequent analysis on them to see if the populations were distinct enough to see if the branching populations were distinct. We tested significance by seeing if the latent feature values were heavily correlated with the axis orthogonal to time. There was only one latent feature that was reclassified to a coherent latent feature in this manner (latent feature 7). Comment the code in the next cell if using a dataset not the MCF10A TGFB1 dataset.

In [ ]:
with open(file_path + "latent_to_bifurcation.json", "r") as file:
    latent_to_bifurcation = json.load(file)
    latent_to_bifurcation = {int(k): v for k, v in latent_to_bifurcation.items()}

## Functional Group Heatmaps

This section regenerates the figures made in the paper.

In [ ]:
def set_pub_style():
    """Sets matplotlib params for publication-quality figures."""
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 7,
        'axes.labelsize': 8,
        'axes.titlesize': 9,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'legend.fontsize': 7,
        'figure.titlesize': 10,
        'axes.linewidth': 0.8,
        'lines.linewidth': 1.5,
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'pdf.fonttype': 42
    })

def plot_dual_mode_emt_figures(
    adata, 
    functional_groups,      
    group_to_latent,        
    latent_to_bifurcation, 
    invert_shap,
    shap_values, 
    gene_names, 
    time_col='timepoint'
):
    set_pub_style()
    
    bifurcating_groups = {}
    non_bifurcating_groups = {}
    
    for grp, lat_idx in group_to_latent.items():
        status = latent_to_bifurcation.get(lat_idx, 0)
        if status == 1:
            bifurcating_groups[grp] = functional_groups[grp]
        else:
            non_bifurcating_groups[grp] = functional_groups[grp]

    # --- Generate Figures ---
    if bifurcating_groups:
        _plot_beam_heatmap(
            adata, bifurcating_groups, group_to_latent, invert_shap, 
            shap_values, gene_names, time_col
        )

    if non_bifurcating_groups:
        _plot_linear_heatmap(
            adata, non_bifurcating_groups, group_to_latent, invert_shap, 
            shap_values, gene_names, time_col
        )

def _plot_beam_heatmap(adata, func_groups, grp_to_lat, invert_shap, shap_vals, genes, time_col):
    ref_grp = list(func_groups.keys())[0]
    ref_lat = grp_to_lat[ref_grp]
    shap_ref = shap_vals[:, :, ref_lat]
    global_signal = np.sum(shap_ref, axis=1)
    if invert_shap.get(ref_lat, 0) == 1: global_signal *= -1
    
    gmm = GaussianMixture(n_components=2, random_state=42)
    labels = gmm.fit_predict(global_signal.reshape(-1, 1))
    if gmm.means_[0] > gmm.means_[1]: high, low = 0, 1
    else: high, low = 1, 0
    adata.obs['branch_id'] = np.where(labels == high, 'Branch_A', 'Branch_B')
    
    scores = _calculate_group_shap_scores(adata, func_groups, grp_to_lat, invert_shap, shap_vals, genes)
    data = pd.concat([adata.obs[[time_col, 'branch_id']], scores], axis=1)
    
    timepoints = sorted(data[time_col].unique())
    root_t = timepoints[0]
    cols, heatmap_vals = [], []
    
    for t in sorted(timepoints, reverse=True):
        if t == root_t: continue
        subset = data[(data[time_col] == t) & (data['branch_id'] == 'Branch_A')]
        if len(subset) > 0:
            heatmap_vals.append(subset[scores.columns].mean())
            cols.append(f"{t}_A")

    subset = data[data[time_col] == root_t]
    heatmap_vals.append(subset[scores.columns].mean())
    cols.append(f"{root_t}_Root")
 
    for t in timepoints:
        if t == root_t: continue
        subset = data[(data[time_col] == t) & (data['branch_id'] == 'Branch_B')]
        if len(subset) > 0:
            heatmap_vals.append(subset[scores.columns].mean())
            cols.append(f"{t}_B")
            
    matrix = pd.DataFrame(heatmap_vals, index=cols).T
    
   
    row_order = []
    for g in matrix.index:
        left = matrix.loc[g, [c for c in cols if '_A' in c]].values
        right = matrix.loc[g, [c for c in cols if '_B' in c]].values
        min_len = min(len(left), len(right))
        diff = np.sum(np.abs(left[:min_len][::-1] - right[:min_len]))
        row_order.append({'g': g, 'd': diff})
    matrix = matrix.loc[pd.DataFrame(row_order).sort_values('d', ascending=False)['g']]
    
  
    height = len(matrix) * 0.25 + 2.0 
    fig, ax = plt.subplots(figsize=(7, height)) # Width = 7 inches
    
    plot_mat = matrix.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    
    sns.heatmap(plot_mat, cmap='coolwarm', center=0, ax=ax, 
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Latent Activity (Z-Score)', 'shrink': 0.5, 'aspect': 10})
    
    center_idx = [i for i, c in enumerate(cols) if 'Root' in c][0]
    ax.axvline(center_idx + 0.5, color='black', ls='--', lw=1.2, alpha=0.8)
    
    ax.set_title("Bifurcating Functional Groups", fontweight='bold', pad=10)
    
    
    raw_days = [c.split('_')[0] for c in cols]
    formatted_days = [d.replace('T', 'Day') if 'T' in d else f"Day{d}" for d in raw_days]
    
    ax.set_xticks(np.arange(len(formatted_days)) + 0.5)
    ax.set_xticklabels(formatted_days, rotation=45, ha='right') # ROTATION ADDED HERE
    
    ax.set_xlabel("← Branch 1          Time          Branch 2 →", labelpad=10)
    
    plt.tight_layout()


def _plot_linear_heatmap(adata, func_groups, grp_to_lat, invert_shap, shap_vals, genes, time_col):
    scores = _calculate_group_shap_scores(adata, func_groups, grp_to_lat, invert_shap, shap_vals, genes)
    data = pd.concat([adata.obs[[time_col]], scores], axis=1)
    
    timepoints = sorted(data[time_col].unique())
    heatmap_vals = []
    for t in timepoints:
        subset = data[data[time_col] == t]
        if len(subset) > 0: heatmap_vals.append(subset[scores.columns].mean())
            
    matrix = pd.DataFrame(heatmap_vals, index=timepoints).T
    
    # Sort
    peak_times = matrix.idxmax(axis=1)
    t_map = {t: i for i, t in enumerate(timepoints)}
    matrix['peak_idx'] = peak_times.map(t_map)
    matrix = matrix.sort_values('peak_idx').drop(columns=['peak_idx'])
    

    height = len(matrix) * 0.25 + 1.8
    fig, ax = plt.subplots(figsize=(7, height)) 
    
    plot_mat = matrix.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    
    sns.heatmap(plot_mat, cmap='coolwarm', center=0, ax=ax,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Latent Activity (Z-Score)', 'shrink': 0.5, 'aspect': 10})
    
    ax.set_title("Non-Bifurcating Functional Groups", fontweight='bold', pad=10)
    ax.set_xlabel("Time", labelpad=5)
    
    current_labels = [label.get_text() for label in ax.get_xticklabels()]
    new_labels = [l.replace('T', 'Day') if 'T' in l else f"Day{l}" for l in current_labels]
    
    ax.set_xticklabels(new_labels, rotation=45, ha='right') # ROTATION ADDED HERE
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    
    plt.tight_layout()

def _calculate_group_shap_scores(adata, func_groups, grp_to_lat, invert_shap, shap_vals, genes):
    scores = pd.DataFrame(index=adata.obs_names)
    gene_map = {name: i for i, name in enumerate(genes)}
    
    for grp, gene_list in func_groups.items():
        if grp not in grp_to_lat: continue
        lat_idx = grp_to_lat[grp]
        g_idxs = [gene_map[g] for g in gene_list if g in gene_map]
        
        if g_idxs:
            val = np.sum(shap_vals[:, g_idxs, lat_idx], axis=1)
            if invert_shap.get(lat_idx, 0) == 1:
                val *= -1
            scores[grp] = val
    return scores

In [ ]:
plot_dual_mode_emt_figures(
    adata_latent, 
    functional_groups,      
    group_to_latent,        
    latent_to_bifurcation,  
    invert_shap,
    shap_values, 
    gene_names, 
    time_col='timepoint'
)

Package version: `sccont.pl.functional_group_heatmap` draws the linear layout for any categorical label and, given a two-valued branch label in `obs` (here from `sccont.pl.assign_branches` on the bifurcating latent), the bifurcating layout.

In [ ]:
bifurcating = {g: genes for g, genes in functional_groups.items() if latent_to_bifurcation.get(group_to_latent[g], 0) == 1}
linear = {g: genes for g, genes in functional_groups.items() if g not in bifurcating}

if bifurcating:
    ref_group = next(iter(bifurcating))
    ref_latent = group_to_latent[ref_group]
    sccont.pl.assign_branches(adata, shap_values, ref_latent, invert=bool(invert_shap.get(ref_latent, 0)))
    sccont.pl.functional_group_heatmap(adata, shap_values, bifurcating, group_to_latent, groupby='timepoint',
                                       invert_shap=invert_shap, branch_key='sccont_branch')
if linear:
    sccont.pl.functional_group_heatmap(adata, shap_values, linear, group_to_latent, groupby='timepoint',
                                       invert_shap=invert_shap);

In [ ]:
def plot_multi_latent_trajectory(
    adata, shap_values, gene_names,
    gene_sets,
    latent_dim_idxs=[0],
    bifurcation_thresholds=[0.75],
    invert_shap_sums=[False],
    latent_colors=['#2A9D8F', '#E76F51', '#E9C46A', '#264653', '#8E9AAF'],
    time_col='timepoint',
    robust_percentiles=(1, 99),
    offset_threshold=None, 
    offset_magnitude=None, 
    background_alpha=0.5
):
    """
    Plots trajectories with black connecting lines and side-by-side offsets.
    """
    params_len = len(latent_dim_idxs)
    
    if len(latent_colors) < params_len:
        latent_colors = (latent_colors * (params_len // len(latent_colors) + 1))
    latent_colors = latent_colors[:params_len]

    markers = ['o', 's', '^', '*', 'p', 'H']

    obs_df = adata.obs.copy()

    X_pca = adata.obsm['X_pca']
    X_pca = X_pca[:, :2]
    
    x_range = X_pca[:, 0].max() - X_pca[:, 0].min()
    y_range = X_pca[:, 1].max() - X_pca[:, 1].min()
    plot_scale = max(x_range, y_range)

    if offset_threshold is None:
        offset_threshold = plot_scale * 0.02 
    if offset_magnitude is None:
        offset_magnitude = plot_scale * 0.01 
    
    timepoints = sorted(obs_df[time_col].unique())
    t_start, t_end = timepoints[0], timepoints[-1]
    c_start = np.mean(X_pca[obs_df[time_col] == t_start], axis=0)
    c_end = np.mean(X_pca[obs_df[time_col] == t_end], axis=0)

    vec_flow = c_end - c_start
    vec_norm = np.linalg.norm(vec_flow)
    if vec_norm > 0: vec_flow /= vec_norm
    else: vec_flow = np.array([1.0, 0.0])

    vec_orth = np.array([-vec_flow[1], vec_flow[0]])
    obs_df['orth_pos'] = np.dot(X_pca - c_start, vec_orth)

    master_trajectories = {}
    norms = {}
    all_norm_vals = np.zeros((adata.shape[0], params_len))

    for i, latent_idx in enumerate(latent_dim_idxs):
        threshold = bifurcation_thresholds[i]
        current_gene_list = gene_sets[i]
        should_invert = invert_shap_sums[i]

        shap_latent = shap_values[:, :, latent_idx]
        shap_df = pd.DataFrame(shap_latent, columns=gene_names)
        sum_shap = shap_df[current_gene_list].sum(axis=1)
        if should_invert: sum_shap *= -1

        temp_key = f'temp_shap_{latent_idx}'
        obs_df[temp_key] = sum_shap.values

        vmin = np.percentile(sum_shap.values, robust_percentiles[0])
        vmax = np.percentile(sum_shap.values, robust_percentiles[1])
        if vmin >= vmax: vmin, vmax = -0.001, 0.001
        current_norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        norms[latent_idx] = current_norm
        all_norm_vals[:, i] = current_norm(sum_shap.values)

        gmm = GaussianMixture(n_components=2, random_state=42)
        gmm.fit(sum_shap.values.reshape(-1, 1))
        high_c, low_c = (0, 1) if gmm.means_[0] > gmm.means_[1] else (1, 0)

        traj_pts = {}

        def get_weighted_centroid(indices, values):
            coords = X_pca[indices]
            w = np.abs(values)
            if np.sum(w) < 1e-9: return np.mean(coords, axis=0)
            return np.average(coords, axis=0, weights=w)

        def make_point(indices, v, p_type):
            return {
                'coords': get_weighted_centroid(indices, v),
                'mean_shap': np.mean(v),
                'type': p_type,
                'latent_idx': latent_idx,
                'marker': markers[i % len(markers)],
                'latent_id': latent_idx 
            }

        for t in timepoints:
            subset = obs_df[obs_df[time_col] == t]
            subset_indices = np.where(obs_df[time_col] == t)[0]
            if len(subset) < 10 or subset['orth_pos'].std() < 1e-5: corr = 0
            else:
                r, _ = pearsonr(subset['orth_pos'], subset[temp_key])
                corr = abs(r)

            is_split = False if t == timepoints[0] else corr > threshold
            points = []
            vals = subset[temp_key].values

            if not is_split:
                points.append(make_point(subset_indices, vals, 'trunk'))
            else:
                labels = gmm.predict(vals.reshape(-1, 1))
                idx_h = subset_indices[labels == high_c]
                if len(idx_h) > 0:
                    points.append(make_point(idx_h, vals[labels == high_c], 'high'))
                idx_l = subset_indices[labels == low_c]
                if len(idx_l) > 0:
                    points.append(make_point(idx_l, vals[labels == low_c], 'low'))
            traj_pts[t] = points

        master_trajectories[latent_idx] = traj_pts

    deviations = np.abs(all_norm_vals - 0.5)
    dominant_indices = np.argmax(deviations, axis=1)
    winning_vals = all_norm_vals[np.arange(adata.shape[0]), dominant_indices]

    cmap = matplotlib.colormaps['coolwarm']

    final_cell_colors = cmap(winning_vals)

    final_trajectory_by_time = {}
    for t in timepoints:
        all_points_at_t = []
        for l_idx in latent_dim_idxs:
            all_points_at_t.extend(master_trajectories[l_idx][t])

        processed_indices = set()
        clusters = []
        for i in range(len(all_points_at_t)):
            if i in processed_indices: continue
            cluster = [i]
            processed_indices.add(i)
            p1 = all_points_at_t[i]
            for j in range(i + 1, len(all_points_at_t)):
                if j in processed_indices: continue
                p2 = all_points_at_t[j]
                if np.linalg.norm(p1['coords'] - p2['coords']) < offset_threshold:
                    cluster.append(j)
                    processed_indices.add(j)
            clusters.append(cluster)
            
        final_points_at_t = []
        for cluster_indices in clusters:
            cluster_points = [all_points_at_t[i] for i in cluster_indices]
            if len(cluster_points) > 1:
                center_coords = np.mean([p['coords'] for p in cluster_points], axis=0)
                cluster_points.sort(key=lambda x: x['latent_id'])
                for k, p in enumerate(cluster_points):
                    angle = (2 * np.pi * k) / len(cluster_points)
                    p['coords'] = center_coords + np.array([offset_magnitude * np.cos(angle), 
                                                           offset_magnitude * np.sin(angle)])
            final_points_at_t.extend(cluster_points)
        final_trajectory_by_time[t] = final_points_at_t

    fig, ax = plt.subplots(figsize=(11, 10))
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=final_cell_colors,
               s=25, alpha=background_alpha, edgecolor='none')

    min_size, max_size = 100, 350
    fixed_node_size = min_size + (3 * (max_size - min_size) / (len(timepoints) - 1)) if len(timepoints) > 3 else 200

    def draw_connection(p_start, p_end):
        line_color = 'black'
        z_val = 10
        
        ax.plot([p_start['coords'][0], p_end['coords'][0]],
                [p_start['coords'][1], p_end['coords'][1]],
                color=line_color, lw=5, zorder=z_val, linestyle='-')

        mid_x = (p_start['coords'][0] + p_end['coords'][0]) / 2
        mid_y = (p_start['coords'][1] + p_end['coords'][1]) / 2
        
        if np.hypot(p_end['coords'][0] - p_start['coords'][0],
                    p_end['coords'][1] - p_start['coords'][1]) > 0.1:
            ax.annotate('',
                        xy=(mid_x, mid_y),
                        xytext=(p_start['coords'][0], p_start['coords'][1]),
                        arrowprops=dict(arrowstyle="->", color=line_color, lw=2,
                                        shrinkA=0, shrinkB=0),
                        zorder=z_val + 5)

    for ti in range(len(timepoints) - 1):
        t_curr, t_next = timepoints[ti], timepoints[ti+1]
        for p2 in final_trajectory_by_time[t_next]:
            l_idx = p2['latent_id']
            best_p1 = None
            min_dist = 999999
            for p1 in final_trajectory_by_time[t_curr]:
                if p1['latent_id'] == l_idx:
                    d = np.linalg.norm(p1['coords'] - p2['coords'])
                    if d < min_dist:
                        min_dist, best_p1 = d, p1
            if best_p1 is not None:
                draw_connection(best_p1, p2)

    for t in timepoints:
        for p in final_trajectory_by_time[t]:
            norm_val = norms[p['latent_id']](p['mean_shap'])
            rgba = cmap(norm_val)
            brightness = (0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2])
            node_color = 'white' if (0.4 < norm_val < 0.6) or (brightness > 0.9) else rgba

            ax.scatter(p['coords'][0], p['coords'][1],
                       color=node_color, s=400, 
                       marker=p['marker'], zorder=30,
                       edgecolors='black', linewidth=1.5)

    legend_handles = [mlines.Line2D([], [], color='black', marker=markers[i % len(markers)],
                                    linestyle='None', markersize=10, label=f'Latent {l_idx}')
                      for i, l_idx in enumerate(latent_dim_idxs)]

    ax.legend(handles=legend_handles, loc='best', frameon=True)
    plt.tight_layout()
    plt.axis('off')
    return fig, ax

In [ ]:
plot_multi_latent_trajectory(adata_emt, 
        shap_values, 
        gene_names,
        [list(latent_0_gene_set), list(latent_5_gene_set), list(latent_6_gene_set), list(latent_7_gene_set)],
        latent_dim_idxs=[0, 5, 6, 7],
        bifurcation_thresholds=[0.7, 0.7, 0.8, 0.7],
        invert_shap_sums=[False, False, False, True])

Package version of the multi-latent trajectory figure (`groupby` can be any ordered label).

In [ ]:
sccont.pl.latent_trajectory(
    adata, shap_values,
    [list(latent_0_gene_set), list(latent_5_gene_set), list(latent_6_gene_set), list(latent_7_gene_set)],
    latents=[0, 5, 6, 7], groupby='timepoint',
    bifurcation_thresholds=[0.7, 0.7, 0.8, 0.7], invert=[False, False, False, True]);

In [ ]:
plot_multi_latent_trajectory(adata_emt, 
        shap_values, 
        gene_names,
        [list(latent_21_gene_set), list(latent_25_gene_set), list(latent_31_gene_set)],
        latent_dim_idxs=[21, 25, 31],
        bifurcation_thresholds=[0.6, 0.6, 0.6],
        invert_shap_sums=[False, False, False], 
        offset_threshold=0.1339,
        offset_magnitude=0.0669)